# <font color="#418FDE" size="6.5" uppercase>**Scaling Distributions**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Apply standard, min-max, max-absolute, robust, and sample normalization transformations. 
- Compare quantile and power transformations for skewed numerical features. 
- Choose scaling strategies that respect sparse and dense data constraints. 


## **1. Core Scaling Methods**

### **1.1. Standard Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_01.jpg?v=1783979917" width="250">



>* Centers values around the feature average
>* Makes differently scaled features more comparable

>* Compare values by relative position
>* Prevent scale dominance in sensitive algorithms

>* Fit scaling on training data only
>* Outliers may require robust alternatives



In [ ]:
#@title Python Code - Standard Scaling

# Demonstrate standard scaling on simple numeric features.
# Compare original units with standardized values.
# Show centered means and unit spreads.

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Create a tiny dataset with very different measurement scales.
housing_data = pd.DataFrame(
    {
        "area_sq_ft": [850, 1200, 1500, 1800, 2400],
        "bedrooms": [2, 3, 3, 4, 5],
        "distance_miles": [1.2, 3.5, 2.1, 6.8, 9.0],
    }
)

# Validate that the example has the expected rectangular shape.
if housing_data.shape != (5, 3):
    raise ValueError("Expected five homes and three numeric features.")

# Fit the scaler on the data and transform every feature.
scaler = StandardScaler()
scaled_values = scaler.fit_transform(housing_data)

# Put scaled values back into a labeled table.
scaled_data = pd.DataFrame(
    scaled_values,
    columns=housing_data.columns,
)

# Summarize the original feature centers and spreads.
original_summary = housing_data.agg(["mean", "std"]).round(2)
scaled_summary = scaled_data.agg(["mean", "std"]).round(2)

print("Original data uses different units and ranges:")
print(housing_data.round(2).to_string(index=False))

print("Scaled data is centered near zero:")
print(scaled_data.round(2).to_string(index=False))

print("Original means and sample standard deviations:")
print(original_summary.to_string())

print("Scaled means and sample standard deviations:")
print(scaled_summary.to_string())



### **1.2. Min Max Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_02.jpg?v=1783979913" width="250">



>* Rescales values to a fixed range
>* Makes different feature units comparable

>* Preserves distribution shape and relative distances
>* Makes features comparable for distance-based models

>* Extreme values can distort min max scaling
>* Fit scaling only on training data



In [ ]:
#@title Python Code - Min Max Scaling

# This example demonstrates min max scaling.
# Each feature becomes bounded between zero and one.
# Printed values show preserved ordering and spacing.

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Small in-memory data keeps the transformation easy to inspect.
raw_data = pd.DataFrame(
    {
        "size_sq_ft": [500, 750, 1000, 1250, 1500],
        "income_usd": [30000, 45000, 60000, 75000, 90000],
        "commute_min": [10, 20, 35, 50, 70],
    }
)

# Validate that every feature has variation before scaling.
feature_ranges = raw_data.max() - raw_data.min()
if (feature_ranges == 0).any():
    raise ValueError("Each feature needs at least two different values.")

# Fit learns each column minimum and maximum.
scaler = MinMaxScaler()
scaled_array = scaler.fit_transform(raw_data)

# Convert the scaled result back to labeled columns.
scaled_data = pd.DataFrame(
    scaled_array,
    columns=raw_data.columns,
)

# Show a compact before-and-after comparison.
comparison = pd.concat(
    [raw_data.add_prefix("raw_"), scaled_data.add_prefix("scaled_")],
    axis=1,
)

print("Min max scaling maps each feature to the range 0 to 1.")
print(comparison.round(2).to_string(index=False))
print("Learned minimums:", np.round(scaler.data_min_, 2).tolist())
print("Learned maximums:", np.round(scaler.data_max_, 2).tolist())



### **1.3. Max Absolute Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_03.jpg?v=1783979915" width="250">



>* Scales values by largest absolute feature value
>* Preserves signs without centering or shifting

>* Keeps sparse data zeros unchanged
>* Scales feature sizes without densifying data

>* Extreme values can compress typical scaled data
>* Best when zeros, signs, and maxima matter



In [ ]:
#@title Python Code - Max Absolute Scaling

# Demonstrate max absolute scaling on signed values.
# Preserve zeros and signs while shrinking magnitudes.
# Compare original and scaled feature ranges.

import numpy as np
import pandas as pd
from sklearn.preprocessing import MaxAbsScaler

# Create small signed features with meaningful zero values.
original_data = pd.DataFrame(
    {"profit_loss_dollars": [-200, -50, 0, 75, 400], "sentiment_score": [-4, 0, 1, 2, 8]}
)

# Validate that the example has the expected small shape.
if original_data.shape != (5, 2):
    raise ValueError("Expected five rows and two numeric features.")

# Fit the scaler and transform each column independently.
scaler = MaxAbsScaler()
scaled_array = scaler.fit_transform(original_data)

# Put scaled values back into a labeled table.
scaled_data = pd.DataFrame(
    scaled_array, columns=["scaled_profit_loss", "scaled_sentiment"]
)

# Combine original and scaled values for easy comparison.
comparison = pd.concat([original_data, scaled_data.round(2)], axis=1)
print(comparison.to_string(index=False))

# Show the largest absolute value used for each feature.
print("Max absolute values used:", scaler.max_abs_.round(2).tolist())

# Confirm that zero values remain exactly zero after scaling.
zero_count = int((scaled_array == 0).sum())
print("Zero values after scaling:", zero_count)



## **2. Robust Transforming**

### **2.1. Robust Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_01.jpg?v=1783979935" width="250">



>* Scales skewed features using median and IQR
>* Limits outlier influence on typical values

>* Robust scaling preserves distribution shape
>* Quantile and power transformations reshape skewed features

>* Use robust scaling for meaningful outliers
>* Choose power or quantile for reshaping



In [ ]:
#@title Python Code - Robust Scaling

# Demonstrate robust scaling on skewed purchase amounts.
# Compare conservative scaling with distribution reshaping.
# Show how outliers affect transformed feature values.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import PowerTransformer

# Create a small skewed feature with realistic large outliers.
purchase_amounts = np.array(
    [18, 22, 25, 27, 30, 34, 38, 42, 48, 55, 70, 95, 180, 420],
    dtype=float,
)

# Validate the feature before applying transformations.
if purchase_amounts.ndim != 1 or purchase_amounts.size < 5:
    raise ValueError("Use one numeric feature with at least five values.")

# Scikit-learn transformers expect a two-dimensional feature matrix.
amount_matrix = purchase_amounts.reshape(-1, 1)

# Robust scaling changes center and scale, not distribution shape.
robust_values = RobustScaler().fit_transform(amount_matrix).ravel()

# Quantile transformation reshapes values by their ranks.
quantile_values = QuantileTransformer(
    n_quantiles=purchase_amounts.size,
    output_distribution="normal",
    random_state=42,
).fit_transform(amount_matrix).ravel()

# Power transformation smoothly reduces skewness when possible.
power_values = PowerTransformer(
    method="yeo-johnson",
    standardize=True,
).fit_transform(amount_matrix).ravel()

# Summarize how each method treats typical and extreme values.
summary = pd.DataFrame(
    {
        "original_$": purchase_amounts,
        "robust": robust_values,
        "quantile_normal": quantile_values,
        "power": power_values,
    }
)

# Print only selected rows to keep the output beginner-friendly.
selected_rows = summary.iloc[[0, 5, 10, 12, 13]].round(2)
print("Selected transformed purchase amounts:")
print(selected_rows.to_string(index=False))

# Plot transformed values against the original feature values.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(purchase_amounts, robust_values, marker="o", label="Robust scaling")
ax.plot(purchase_amounts, quantile_values, marker="o", label="Quantile normal")
ax.plot(purchase_amounts, power_values, marker="o", label="Power transform")

# Label the plot to compare scaling and reshaping effects.
ax.set_title("Robust Scaling Versus Distribution Reshaping")
ax.set_xlabel("Original purchase amount ($)")
ax.set_ylabel("Transformed value")
ax.legend()
plt.show()



### **2.2. Sample Normalization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_02.jpg?v=1783979937" width="250">



>* Rescales each row, not each feature
>* Compares patterns instead of document size

>* Quantile and power reshape skewed features
>* Sample normalization compares within-sample proportions

>* Normalize samples only for relative composition
>* Use quantile or power for feature skew



In [ ]:
#@title Python Code - Sample Normalization

# Compare row normalization with feature transformations.
# Sample normalization rescales each observation profile.
# The output shows changed row magnitudes.

import numpy as np
import pandas as pd
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import QuantileTransformer

# Each row is one customer spending profile.
spending = np.array(
    [[20, 5, 0], [200, 50, 0], [10, 80, 10], [100, 800, 100]],
    dtype=float,
)

# Validate the small teaching array before transforming.
if spending.shape != (4, 3):
    raise ValueError("Expected four customers and three categories.")

# Sample normalization makes each row length equal to one.
row_normalizer = Normalizer(norm="l2")
row_scaled = row_normalizer.transform(spending)

# Feature transformations reshape each column across customers.
quantile = QuantileTransformer(
    n_quantiles=4, output_distribution="normal", random_state=42
)
power = PowerTransformer(method="yeo-johnson", standardize=True)

quantile_scaled = quantile.fit_transform(spending)
power_scaled = power.fit_transform(spending)

# Compare row magnitudes after each transformation.
summary = pd.DataFrame(
    {
        "raw": np.linalg.norm(spending, axis=1),
        "sample_norm": np.linalg.norm(row_scaled, axis=1),
        "quantile": np.linalg.norm(quantile_scaled, axis=1),
        "power": np.linalg.norm(power_scaled, axis=1),
    }
)

summary.index = ["small same mix", "large same mix", "grocery", "large grocery"]
summary = summary.round(2)

print("Row magnitude comparison: sample normalization controls row size.")
print(summary.to_string())



### **2.3. Sparse Safe Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_03.jpg?v=1783979938" width="250">



>* Preserve meaningful zeros in sparse data
>* Avoid transformations that make data dense

>* Quantile transforms can distort sparse zeros
>* Power transforms may not preserve sparsity

>* Scale nonzeros without centering sparse data
>* Preserve zero meaning while transforming positives



In [ ]:
#@title Python Code - Sparse Safe Scaling

# This example compares sparse safe scaling choices.
# Multiplicative scaling preserves meaningful zero entries.
# Centering and quantile transforms can densify data.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import QuantileTransformer

# Rows mimic customers, and columns mimic rare event counts.
counts = np.array(
    [[0, 0, 1], [0, 2, 0], [0, 0, 8], [3, 0, 0], [0, 0, 0]],
    dtype=float,
)

# This check keeps the tiny teaching matrix understandable.
if counts.shape != (5, 3):
    raise ValueError("Expected five rows and three features.")

# Count how many stored values would be nonzero.
original_nonzero = np.count_nonzero(counts)

# MaxAbsScaler only multiplies, so zeros stay zero.
maxabs_values = MaxAbsScaler().fit_transform(counts)
maxabs_nonzero = np.count_nonzero(maxabs_values)

# Centering subtracts medians, which can create new nonzero values.
robust_values = RobustScaler(with_centering=True).fit_transform(counts)
robust_nonzero = np.count_nonzero(robust_values)

# QuantileTransformer changes ranks and may also change zero entries.
quantile = QuantileTransformer(
    n_quantiles=5, output_distribution="normal", random_state=42
)
quantile_values = quantile.fit_transform(counts)
quantile_nonzero = np.count_nonzero(np.round(quantile_values, 12))

print("Nonzero entries before scaling:", original_nonzero)
print("After MaxAbsScaler:", maxabs_nonzero)
print("After centered RobustScaler:", robust_nonzero)
print("After QuantileTransformer:", quantile_nonzero)
print("First feature after quantile:", np.round(quantile_values[:, 0], 2))

# The bar chart highlights which methods preserve sparsity.
methods = ["Original", "MaxAbs", "Robust", "Quantile"]
nonzero_counts = [original_nonzero, maxabs_nonzero, robust_nonzero, quantile_nonzero]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(methods, nonzero_counts, color=["gray", "green", "orange", "red"])
ax.set_title("Sparse Safety: How Many Entries Become Nonzero?")
ax.set_xlabel("Transformation")
ax.set_ylabel("Number of nonzero entries")
ax.set_ylim(0, counts.size)
plt.show()



## **3. Distribution Shape**

### **3.1. Quantile Mapping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_01.jpg?v=1783979911" width="250">



>* Maps values by ranked position
>* Handles skewed features and extreme values

>* Dense features usually handle quantile mapping well
>* Sparse zeros may lose meaning and efficiency

>* Use quantile mapping for skewed dense features
>* Preserve zeros in sparse feature data



In [ ]:
#@title Python Code - Quantile Mapping

# This example compares dense and sparse quantile mapping.
# Quantile mapping reshapes ranks into target distributions.
# The output highlights when sparsity is preserved.

import numpy as np
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import MaxAbsScaler

# A fixed generator makes the dense example reproducible.
rng = np.random.default_rng(42)
dense_spending = rng.lognormal(mean=3.0, sigma=1.0, size=200)

# Many zeros represent absence in this sparse-like feature.
sparse_counts = np.zeros(200)
sparse_counts[:20] = rng.integers(1, 8, size=20)

# Quantile mapping is fitted separately to each feature.
quantile_mapper = QuantileTransformer(
    n_quantiles=50,
    output_distribution="uniform",
    random_state=42,
)

# Max-absolute scaling keeps zeros exactly zero.
max_abs_scaler = MaxAbsScaler()

# Dense skewed values become evenly spread by rank.
dense_quantiles = quantile_mapper.fit_transform(
    dense_spending.reshape(-1, 1)
).ravel()

# Sparse-like zeros can become nonzero after quantile mapping.
sparse_quantiles = quantile_mapper.fit_transform(
    sparse_counts.reshape(-1, 1)
).ravel()

# Max-absolute scaling preserves the absence pattern.
sparse_max_abs = max_abs_scaler.fit_transform(
    sparse_counts.reshape(-1, 1)
).ravel()

# Validate that each transformed feature kept the original length.
if len(dense_quantiles) != len(dense_spending):
    raise ValueError("Dense transformed data has the wrong length.")

if len(sparse_quantiles) != len(sparse_counts):
    raise ValueError("Sparse transformed data has the wrong length.")

# Count nonzero entries to show the representation effect.
original_nonzero = np.count_nonzero(sparse_counts)
quantile_nonzero = np.count_nonzero(sparse_quantiles)
max_abs_nonzero = np.count_nonzero(sparse_max_abs)

print("Dense spending before: min={:.1f}, median={:.1f}, max={:.1f}".format(
    dense_spending.min(), np.median(dense_spending), dense_spending.max()
))

print("Dense spending after quantile mapping: min={:.2f}, median={:.2f}, max={:.2f}".format(
    dense_quantiles.min(), np.median(dense_quantiles), dense_quantiles.max()
))

print("Sparse-like feature nonzeros before scaling: {} of 200".format(
    original_nonzero
))

print("After quantile mapping, nonzeros: {} of 200".format(
    quantile_nonzero
))

print("After max-absolute scaling, nonzeros: {} of 200".format(
    max_abs_nonzero
))

print("Lesson: quantile mapping suits dense skewed features, not structural zeros.")



### **3.2. Power Transform**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_02.jpg?v=1783979910" width="250">



>* Reshapes skewed data toward balanced distributions
>* Reduces extreme influence while preserving order

>* Dense features usually suit power transforms
>* Sparse zeros can become costly nonzeros

>* Match transforms to skew and data structure
>* Preserve zeros, sparsity, meaning, and efficiency



In [ ]:
#@title Python Code - Power Transform

# This example compares dense and sparse preprocessing choices.
# Power transforms reshape skewed dense numerical features.
# Sparse zeros should usually remain stored as zeros.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import MaxAbsScaler

# Create one skewed dense feature with deterministic values.
rng = np.random.default_rng(42)
dense_amounts = rng.lognormal(mean=2.0, sigma=1.0, size=300)

# Create a sparse-like count feature with many meaningful zeros.
sparse_counts = np.zeros(300)
nonzero_positions = rng.choice(300, size=30, replace=False)
sparse_counts[nonzero_positions] = rng.integers(1, 8, size=30)

# Validate the small teaching arrays before transforming them.
if dense_amounts.size != sparse_counts.size:
    raise ValueError("Both features must have the same row count.")

# PowerTransformer expects a two-dimensional feature matrix.
dense_matrix = dense_amounts.reshape(-1, 1)
power_transformer = PowerTransformer(method="yeo-johnson", standardize=True)
dense_power = power_transformer.fit_transform(dense_matrix).ravel()

# MaxAbsScaler preserves zeros, which is useful for sparse-style data.
sparse_matrix = sparse_counts.reshape(-1, 1)
max_abs_scaler = MaxAbsScaler()
sparse_scaled = max_abs_scaler.fit_transform(sparse_matrix).ravel()

# Count zeros to show whether sparse structure was preserved.
original_zero_count = int(np.sum(sparse_counts == 0))
scaled_zero_count = int(np.sum(sparse_scaled == 0))

print("Dense feature skew before power transform:", round(float(np.mean(dense_amounts)), 2))
print("Dense feature mean after power transform:", round(float(np.mean(dense_power)), 2))
print("Sparse-style zeros before MaxAbs scaling:", original_zero_count)
print("Sparse-style zeros after MaxAbs scaling:", scaled_zero_count)
print("Lesson: use power transforms for dense skew, not sparse absence signals.")

# Plot the dense feature before and after the power transform.
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(dense_amounts, bins=30, alpha=0.55, label="Original dense values")
ax.hist(dense_power, bins=30, alpha=0.55, label="Power transformed values")
ax.set_title("Power Transform Changes Dense Distribution Shape")
ax.set_xlabel("Feature value")
ax.set_ylabel("Number of rows")
ax.legend()
plt.show()



### **3.3. Distribution Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_03.jpg?v=1783979907" width="250">



>* Quantile mapping strongly reshapes skewed features
>* Power transforms smooth skew while preserving relationships

>* Match transformations to dense or sparse data
>* Preserve zeros with sparse-safe scaling methods

>* Compare transformations within the full pipeline
>* Preserve meaning, sparsity, memory, and performance



In [ ]:
#@title Python Code - Distribution Comparison

# Compare dense and sparse scaling choices.
# Distribution transforms can change stored zeros.
# The output highlights sparsity preservation.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import PowerTransformer

# Build a tiny sparse-like feature with many exact zeros.
spending = np.array([[0], [0], [0], [1], [2], [5], [20], [100]], dtype=float)

# Validate the example shape before transforming.
if spending.shape != (8, 1):
    raise ValueError("Expected one feature with eight rows.")

# Apply transformations that handle distribution shape differently.
max_abs_values = MaxAbsScaler().fit_transform(spending)
quantile_values = QuantileTransformer(
    output_distribution="normal", n_quantiles=8, random_state=42
).fit_transform(spending)
power_values = PowerTransformer(method="yeo-johnson").fit_transform(spending)

# Count how many transformed entries remain exactly zero.
raw_zero_count = int(np.sum(spending == 0))
max_abs_zero_count = int(np.sum(max_abs_values == 0))
quantile_zero_count = int(np.sum(quantile_values == 0))
power_zero_count = int(np.sum(power_values == 0))

print("Zero counts out of 8 values:")
print(f"raw={raw_zero_count}, max_abs={max_abs_zero_count}")
print(f"quantile_normal={quantile_zero_count}, power={power_zero_count}")
print("MaxAbs keeps zeros; distribution shaping usually densifies them.")

# Plot raw values against transformed values for comparison.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(spending.ravel(), max_abs_values.ravel(), label="MaxAbs")
ax.scatter(spending.ravel(), quantile_values.ravel(), label="Quantile normal")
ax.scatter(spending.ravel(), power_values.ravel(), label="Power")

ax.set_title("Distribution transforms versus sparsity preservation")
ax.set_xlabel("Original spending amount")
ax.set_ylabel("Transformed value")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Scaling Distributions**</font>


In this lecture, you learned to:
- Apply standard, min-max, max-absolute, robust, and sample normalization transformations. 
- Compare quantile and power transformations for skewed numerical features. 
- Choose scaling strategies that respect sparse and dense data constraints. 

In the next Lecture (Lecture B), we will go over 'Expansion Binning'